# Diffusion Sources: Colab GPU Training
Drive stores datasets and checkpoints; GitHub provides versioned code.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/diffusion-sources'
REPOSITORY = 'https://github.com/1habibi/diffusion-sources-localization-.git'


In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime first'
print(torch.__version__, torch.version.cuda, torch.cuda.get_device_name(0))


In [ ]:
from pathlib import Path
repo = Path('/content/diffusion-sources')
if not repo.exists():
    !git clone {REPOSITORY} /content/diffusion-sources
else:
    %cd /content/diffusion-sources
    !git pull --ff-only
%cd /content/diffusion-sources
!pip install -q networkx numpy matplotlib PyYAML scikit-learn streamlit tqdm torch-geometric
!pip install -q -e . --no-deps
!git rev-parse HEAD


In [ ]:
from pathlib import Path
import yaml
base = yaml.safe_load(Path('configs/train_facebook.yaml').read_text())
base['data']['directory'] = f'{DRIVE_ROOT}/data/facebook_main'
base['training']['device'] = 'cuda'
base['training']['seed'] = 7026
base['training']['resume'] = True
Path('/content/train_joint.yaml').write_text(yaml.safe_dump(base, sort_keys=False))
print(yaml.safe_dump(base, sort_keys=False))


## Перед полным обучением
Проверь датасет и выполни первый запуск. Обучения запускаются по одному: после каждого проверяй `metrics.json`. `resume: true` позволяет повторить ту же команду после разрыва Colab.

In [ ]:
from pathlib import Path
DATA = Path(DRIVE_ROOT) / 'data/facebook_main'
required = ['graph.npz', 'train.npz', 'validation.npz', 'test.npz']
assert all((DATA / name).exists() for name in required), f'Missing dataset files in {DATA}'
for name in required:
    path = DATA / name
    print(name, round(path.stat().st_size / 1024**2, 2), 'MiB')
print('Dataset check passed:', DATA)


In [ ]:
import json
from pathlib import Path

def show_result(output_dir, method='joint_estimated_k'):
    result_dir = Path(output_dir)
    metrics = json.loads((result_dir / 'metrics.json').read_text())
    print('best epoch:', metrics['best_epoch'])
    print('stopped epoch:', metrics['stopped_epoch'])
    print('stop reason:', metrics['stop_reason'])
    print('training hours:', round(metrics['training_seconds'] / 3600, 2))
    print('peak VRAM GiB:', round(metrics['peak_memory_bytes'] / 1024**3, 2))
    print(json.dumps(metrics['prediction_metrics'][method]['all'], indent=2))
    print('runtime:', json.dumps(metrics['runtime'], indent=2))
    for name in ['last_checkpoint.pt', 'best_model.pt', 'history.json', 'metrics.json', 'test_predictions.csv']:
        print(name, (result_dir / name).exists())


## Основные запуски
Запускай следующие ячейки последовательно, но не объединяй их в одну очередь. В каждой используется отдельная папка на Drive. Сначала заверши Joint full для трех seed, затем no-consistency и Node-only.

In [ ]:
# Joint full: seed 7026
OUTPUT = f'{DRIVE_ROOT}/reports/joint_full/seed_7026'
!python scripts/train_model.py --config /content/train_joint.yaml --output {OUTPUT}


In [ ]:
show_result(f'{DRIVE_ROOT}/reports/joint_full/seed_7026')


In [ ]:
# Helper: create a config for another seed and model
def make_config(source_config, output_name, seed):
    config = yaml.safe_load(Path(source_config).read_text())
    config['data']['directory'] = f'{DRIVE_ROOT}/data/facebook_main'
    config['training']['device'] = 'cuda'
    config['training']['seed'] = seed
    config['training']['resume'] = True
    path = Path('/content') / f'{output_name}_{seed}.yaml'
    path.write_text(yaml.safe_dump(config, sort_keys=False))
    return str(path)

joint_7027 = make_config('configs/train_facebook.yaml', 'train_joint', 7027)
joint_7028 = make_config('configs/train_facebook.yaml', 'train_joint', 7028)
no_cons_7026 = make_config('configs/train_facebook_no_consistency.yaml', 'train_no_consistency', 7026)
no_cons_7027 = make_config('configs/train_facebook_no_consistency.yaml', 'train_no_consistency', 7027)
no_cons_7028 = make_config('configs/train_facebook_no_consistency.yaml', 'train_no_consistency', 7028)
node_7026 = make_config('configs/train_node_facebook.yaml', 'train_node', 7026)
node_7027 = make_config('configs/train_node_facebook.yaml', 'train_node', 7027)
node_7028 = make_config('configs/train_node_facebook.yaml', 'train_node', 7028)
print('Configs created')


### Joint full: seed 7027 и 7028

In [ ]:
OUTPUT = f'{DRIVE_ROOT}/reports/joint_full/seed_7027'
!python scripts/train_model.py --config {joint_7027} --output {OUTPUT}


In [ ]:
show_result(f'{DRIVE_ROOT}/reports/joint_full/seed_7027')


In [ ]:
OUTPUT = f'{DRIVE_ROOT}/reports/joint_full/seed_7028'
!python scripts/train_model.py --config {joint_7028} --output {OUTPUT}


In [ ]:
show_result(f'{DRIVE_ROOT}/reports/joint_full/seed_7028')


### Joint без consistency-loss

In [ ]:
OUTPUT = f'{DRIVE_ROOT}/reports/no_consistency/seed_7026'
!python scripts/train_model.py --config {no_cons_7026} --output {OUTPUT}
show_result(OUTPUT)


In [ ]:
OUTPUT = f'{DRIVE_ROOT}/reports/no_consistency/seed_7027'
!python scripts/train_model.py --config {no_cons_7027} --output {OUTPUT}
show_result(OUTPUT)


In [ ]:
OUTPUT = f'{DRIVE_ROOT}/reports/no_consistency/seed_7028'
!python scripts/train_model.py --config {no_cons_7028} --output {OUTPUT}
show_result(OUTPUT)


### Node-only

In [ ]:
OUTPUT = f'{DRIVE_ROOT}/reports/node_only/seed_7026'
!python scripts/train_node_model.py --config {node_7026} --output {OUTPUT}
show_result(OUTPUT, 'node_thresholded')


In [ ]:
OUTPUT = f'{DRIVE_ROOT}/reports/node_only/seed_7027'
!python scripts/train_node_model.py --config {node_7027} --output {OUTPUT}
show_result(OUTPUT, 'node_thresholded')


In [ ]:
OUTPUT = f'{DRIVE_ROOT}/reports/node_only/seed_7028'
!python scripts/train_node_model.py --config {node_7028} --output {OUTPUT}
show_result(OUTPUT, 'node_thresholded')


## После всех запусков
Скачай или оставь на Drive `metrics.json`, `history.csv`, `history.json`, `best_model.pt` и `test_predictions.csv` для каждого из девяти каталогов. Затем запусти локальные reporting/series-скрипты или пришли эти метрики для построения сводных таблиц и графиков.